import numpy as np
import pandas as pd

interactions = pd.read_csv("data_original/interactions.csv")
interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])

interactions.shape# Historical Replay

Формируем обучающие выборки для рекомендательного ранжирования на KION.

Для каждой недели:
1. Используем всю историю взаимодействий до начала недели.
2. Генерируем RRF@500 кандидатов.
3. Оставляем warm-start пользователей с новыми просмотрами в прогнозируемую неделю.
4. Размечаем кандидатов по наблюдаемым просмотрам.

Горизонт предсказания — 7 дней.

Первые обучающие примеры формируем на 12–18 апреля 2021 года.

In [ ]:
import numpy as np
import pandas as pd

interactions = pd.read_csv("data_original/interactions.csv")
interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])

interactions.shape

In [ ]:
from pathlib import Path
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import BM25Recommender
import pyarrow.parquet as pq
import pyarrow as pa

def rrf(popular, collaborative, k=60, n=500):
    scores = {}

    for rank, item in enumerate(popular, 1):
        scores[item] = scores.get(item, 0) + 1 / (k + rank)

    for rank, item in enumerate(collaborative, 1):
        scores[item] = scores.get(item, 0) + 1 / (k + rank)

    return sorted(scores, key=scores.get, reverse=True)[:n]

import pyarrow as pa
import pyarrow.parquet as pq


def build_replay(start):
    start = pd.Timestamp(start)
    end = start + pd.Timedelta(days=7)

    history = interactions[interactions["last_watch_dt"] < start]
    future = interactions[
        (interactions["last_watch_dt"] >= start) &
        (interactions["last_watch_dt"] < end)
    ]

    seen = history.groupby("user_id")["item_id"].agg(set)
    future_items = future.groupby("user_id")["item_id"].agg(set)

    targets = {
        u: items - seen[u]
        for u, items in future_items.items()
        if u in seen.index
    }
    targets = {u: items for u, items in targets.items() if items}
    target_users = list(targets)

    popular = (
        history.groupby("item_id")["user_id"]
        .nunique()
        .sort_values(ascending=False)
        .index.to_numpy()
    )

    user_codes, user_ids = pd.factorize(history["user_id"])
    item_codes, item_ids = pd.factorize(history["item_id"])

    matrix = csr_matrix(
        (np.ones(len(history), dtype=np.float32), (user_codes, item_codes)),
        shape=(len(user_ids), len(item_ids))
    )

    model = BM25Recommender(K=200)
    model.fit(matrix, show_progress=False)

    user_index = pd.Series(np.arange(len(user_ids)), index=user_ids)
    indices = user_index.loc[target_users].to_numpy()

    bm25_ids, bm25_scores = model.recommend(
        indices,
        matrix[indices],
        N=500,
        filter_already_liked_items=True
    )

    rows = []

    for u, rec_ids, rec_scores in zip(target_users, bm25_ids, bm25_scores):
        watched = seen[u]

        pop_items = [i for i in popular if i not in watched][:500]

        valid = rec_ids >= 0
        bm25_items = item_ids[rec_ids[valid]]
        scores = rec_scores[valid]

        filtered = [
            (item, score)
            for item, score in zip(bm25_items, scores)
            if item not in watched
        ]

        pop_ranks = {item: rank for rank, item in enumerate(pop_items, 1)}
        bm25_ranks = {
            item: rank
            for rank, (item, _) in enumerate(filtered, 1)
        }
        bm25_values = dict(filtered)

        candidates = rrf(pop_items, [item for item, _ in filtered])

        for rank, item in enumerate(candidates, 1):
            pop_rank = pop_ranks.get(item)
            bm25_rank = bm25_ranks.get(item)

            rrf_score = 0
            if pop_rank is not None:
                rrf_score += 1 / (60 + pop_rank)
            if bm25_rank is not None:
                rrf_score += 1 / (60 + bm25_rank)

            rows.append((
                u, item, int(item in targets[u]),
                pop_rank, bm25_rank, bm25_values.get(item),
                rrf_score, rank
            ))

    samples = pd.DataFrame(
        rows,
        columns=[
            "user_id", "item_id", "label",
            "pop_rank", "bm25_rank", "bm25_score",
            "rrf_score", "rrf_rank"
        ]
    )
    samples["prediction_date"] = start

    path = Path("data_processed/replay") / f"{start.date()}.parquet"
    path.parent.mkdir(parents=True, exist_ok=True)

    pq.write_table(pa.Table.from_pandas(samples, preserve_index=False), path)

    hits = samples.groupby("user_id")["label"].sum()
    recall = np.mean([hits[u] / len(targets[u]) for u in target_users])

    print(f"\n{start.date()}")
    print(f"Пользователей: {len(target_users):,}")
    print(f"Кандидатов: {len(samples):,}")
    print(f"Положительных: {samples['label'].sum():,}")
    print(f"Recall@500: {recall:.4f}")
    print(f"Без положительных кандидатов: {(hits == 0).mean():.2%}")

    return samples

In [ ]:
from tqdm.auto import tqdm

weeks = pd.date_range("2021-04-12", "2021-08-16", freq="7D")

for week in tqdm(weeks, desc="Historical replay"):
    build_replay(week)